In [1]:
!pip install pytorch-crf

In [ ]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from tqdm import tqdm
import fasttext
from torchcrf import CRF
from sklearn.metrics import classification_report, f1_score
import torch
import torch.nn as nn
from torchcrf import CRF

In [3]:
train_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/train.parquet")
val_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/val.parquet")
test_df=pd.read_parquet("/kaggle/input/datasets/karimmahmoud09/pii-splited-data/test.parquet")

In [ ]:
train_df.head()["tokenised_unmasked_text"]

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17330 entries, 0 to 17329
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   masked_text              17330 non-null  object
 1   unmasked_text            17330 non-null  object
 2   token_entity_labels      17330 non-null  object
 3   tokenised_unmasked_text  17330 non-null  object
dtypes: object(4)
memory usage: 541.7+ KB


In [6]:
print(len(train_df), len(val_df),len(test_df)) 

17330 1926 2140


In [7]:
ENWE = fasttext.load_model("/kaggle/input/datasets/bobazooba/fasttext-english/cc.en.300.bin")

In [8]:
all_labels = set()

for labels in train_df["token_entity_labels"]:
    all_labels.update(labels)

label2id = {label: idx for idx, label in enumerate(sorted(all_labels))}
id2label = {idx: label for label, idx in label2id.items()}

print(label2id)
print("Number of classes:", len(label2id))

{'B-ACCOUNTNAME': 0, 'B-ACCOUNTNUMBER': 1, 'B-CREDITCARDNUMBER': 2, 'B-EMAIL': 3, 'B-IPV4': 4, 'B-IPV6': 5, 'B-MAC': 6, 'B-PASSWORD': 7, 'B-PHONE_NUMBER': 8, 'B-SSN': 9, 'B-USERNAME': 10, 'I-ACCOUNTNAME': 11, 'I-ACCOUNTNUMBER': 12, 'I-CREDITCARDNUMBER': 13, 'I-EMAIL': 14, 'I-IPV4': 15, 'I-IPV6': 16, 'I-MAC': 17, 'I-PASSWORD': 18, 'I-PHONE_NUMBER': 19, 'I-SSN': 20, 'I-USERNAME': 21, 'O': 22}
Number of classes: 23


In [ ]:
class PiiDataset(Dataset):

    def __init__(self, texts, labels, embedding_model, label2id):
        self.embedding_model = embedding_model
        self.label2id = label2id
        self.data = [(t, l) for t, l in zip(texts, labels) if len(t) > 0 and len(l) > 0]
        self.max_len = max(len(sent) for sent, _ in self.data)
        self.embed_dim = embedding_model.get_dimension()
        self.pad_embedding = torch.zeros(self.embed_dim)

        self.pad_label = -100

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        words, labels = self.data[idx]
        embeddings = [torch.tensor(self.embedding_model.get_word_vector(word),dtype=torch.float32) for word in words]
        encoded_labels = [self.label2id[label] for label in labels ]
        length = len(words)
        while len(embeddings) < self.max_len:
            embeddings.append(self.pad_embedding)

        while len(encoded_labels) < self.max_len:
            encoded_labels.append(self.pad_label)

        x = torch.stack(embeddings)
        y = torch.tensor(encoded_labels, dtype=torch.long)
        return x, length, y

In [ ]:
training_data = PiiDataset(np.array(train_df["tokenised_unmasked_text"]), np.array(train_df["token_entity_labels"]),embedding_model=ENWE,label2id=label2id)
val_data = PiiDataset(np.array(val_df["tokenised_unmasked_text"]),np.array(val_df["token_entity_labels"]),embedding_model=ENWE,label2id=label2id)
test_data = PiiDataset(np.array(test_df["tokenised_unmasked_text"]),np.array(test_df["token_entity_labels"]),embedding_model=ENWE,label2id=label2id)

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(self,embedding_dim=300,hidden_size=256,num_classes=2,dropout=0.3):
        super().__init__()
        self.bilstm1 = nn.LSTM(embedding_dim,hidden_size,batch_first=True,bidirectional=True)
        self.bilstm2 = nn.LSTM(hidden_size * 2,hidden_size,batch_first=True,bidirectional=True)
        self.bilstm3 = nn.LSTM(hidden_size * 2,hidden_size,batch_first=True,bidirectional=True)

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(nn.Linear(hidden_size * 2, 512),
                                        nn.ReLU(),
                                        nn.Dropout(dropout),
                                        nn.Linear(512, num_classes))

        self.crf = CRF(num_tags=num_classes,batch_first=True)

    def forward(self, x):
        out1, _ = self.bilstm1(x)
        out2, _ = self.bilstm2(out1)
        out3, _ = self.bilstm3(out2)
        emissions = self.classifier(out3)
        return emissions

    def loss(self, x, tags, mask):
        emissions = self.forward(x)
        # beye7seb el negative log likelihood we benesta5dem mean 3ashan ne aggreagate el loss 3ala kol el batch
        loss = -self.crf(emissions,tags,mask=mask,reduction="mean")
        return loss

    def decode(self, x, mask):
        emissions = self.forward(x)
        prediction = self.crf.decode(emissions,mask=mask)
        return prediction

In [ ]:
def overfit_one_batch(model,dataset,batch_size=1,epochs=500,lr=1e-3):
    loader = DataLoader(dataset,batch_size=batch_size,shuffle=True)
    x, lengths, y = next(iter(loader))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    x = x.to(device)
    y = y.to(device)
    lengths = lengths.to(device)

    optimizer = torch.optim.Adam(model.parameters(),lr=lr)
    model.train()

    mask = (y != -100).bool() # create mask
    safe_labels = y.clone() 
    safe_labels[~mask] = 0 # change mask from -100 le 0 3ashan crf doesnt accept mask with -100
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = model.loss(x,safe_labels,mask)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds = model.decode(x,mask)
            correct = 0
            total = 0

            for i in range(len(preds)):
                pred = torch.tensor(preds[i],device=device)
                true = safe_labels[i][mask[i]] # select only valid values 3ashan el crf bey decode el valid values bas
                correct += (pred == true).sum().item()
                total += len(true)

            acc = correct / total
            unique_preds = set()
            for seq in preds:
                unique_preds.update(seq)

        if epoch % 10 == 0:
            print(f"Epoch {epoch} Loss: {loss.item():.4f} Acc: {acc:.4f} Pred labels: {unique_preds}")

In [ ]:
model = BiLSTMCRF(embedding_dim=300,hidden_size=256,num_classes=len(label2id),dropout=0)
overfit_one_batch(model,training_data,batch_size=32,epochs=1000,lr=1e-3)

In [ ]:
model = BiLSTMCRF(embedding_dim=300,hidden_size=256,num_classes=len(label2id),dropout=0.3)

In [ ]:
def evaluate(model, val_dataloader, pad_label=-100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.eval()
    total_correct = 0
    total_tokens = 0
    total_loss = 0

    with torch.no_grad():
        for x, lengths, y in tqdm(val_dataloader):
            x = x.to(device)
            y = y.to(device)
            mask = (y != pad_label)
            safe_labels = y.clone()
            safe_labels[~mask] = 0
            loss = model.loss(x,safe_labels,mask)
            total_loss += loss.item()
            preds = model.decode(x,mask)

            for i in range(len(preds)):
                pred = torch.tensor(preds[i],device=device)
                true = safe_labels[i][:len(pred)]
                total_correct += (pred == true).sum().item()
                total_tokens += len(pred)

    val_loss = total_loss / len(val_dataloader)
    val_acc = total_correct / total_tokens

    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    return val_loss, val_acc

In [ ]:
def train(model,train_dataset,val_dataset,batch_size=64,epochs=20,learning_rate=1e-3,pad_label=-100,patience=3):
    train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False)

    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=2,gamma=0.5)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total_tokens = 0

        for x, lengths, y in tqdm(train_loader):
            x = x.to(device)
            y = y.to(device)
            lengths = lengths.to(device)

            mask = (y != pad_label).bool()
            safe_labels = y.clone()
            safe_labels[~mask] = 0

            optimizer.zero_grad()
            loss = model.loss(x,safe_labels,mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5)
            optimizer.step()
            total_loss += loss.item()

            with torch.no_grad():
                preds = model.decode(x,mask)
                for i in range(len(preds)):
                    pred = torch.tensor(preds[i],device=device)
                    true = safe_labels[i][mask[i]]

                    total_correct += (pred == true).sum().item()
                    total_tokens += len(true)

        scheduler.step()
        train_loss = total_loss / len(train_loader)
        train_acc = total_correct / total_tokens

        val_loss, val_acc = evaluate(model,val_loader,pad_label)

        print( f"Epoch {epoch+1} | Train Loss: {train_loss} | Train Acc: {train_acc} | Val Loss: {val_loss} Val Acc: {val_acc}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(),"bilstm_crf.pth")

        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping")
                model.load_state_dict(torch.load("bilstm_crf.pth",map_location=device))
                break

    return model

In [ ]:
trained_model = train(model=model,train_dataset=training_data,val_dataset=val_data,batch_size=64,epochs=20,learning_rate=1e-3,pad_label=-100,patience=5)

100%|██████████| 31/31 [00:10<00:00,  2.88it/s]



Val Loss: 4.8517 | Val Acc: 0.9663
Epoch 1 | Train Loss: 15.9368 | Train Acc: 0.9461 | Val Loss: 4.8517 | Val Acc: 0.9663


100%|██████████| 31/31 [00:10<00:00,  2.91it/s]



Val Loss: 2.8798 | Val Acc: 0.9742
Epoch 2 | Train Loss: 3.4299 | Train Acc: 0.9753 | Val Loss: 2.8798 | Val Acc: 0.9742


100%|██████████| 31/31 [00:10<00:00,  2.86it/s]


...


Val Loss: 1.0991 | Val Acc: 0.9873
Epoch 12 | Train Loss: 0.5101 | Train Acc: 0.9908 | Val Loss: 1.0991 | Val Acc: 0.9873


100%|██████████| 31/31 [00:10<00:00,  2.90it/s]



Val Loss: 1.1812 | Val Acc: 0.9869
Epoch 13 | Train Loss: 0.4880 | Train Acc: 0.9916 | Val Loss: 1.1812 | Val Acc: 0.9869


100%|██████████| 31/31 [00:10<00:00,  2.91it/s]


Val Loss: 1.1756 | Val Acc: 0.9872
Epoch 14 | Train Loss: 0.4820 | Train Acc: 0.9915 | Val Loss: 1.1756 | Val Acc: 0.9872
Early stopping


In [ ]:
def evaluate_test_set(model,test_dataset,label2id,batch_size=64,pad_label=-100):
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model = model.to(device)
    model.eval()

    loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)
    id2label = {v: k for k, v in label2id.items()}

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, lengths, y in tqdm(loader):
            x = x.to(device)
            y = y.to(device)
            mask = (y != pad_label)
            preds = model.decode(x,mask)

            for i in range(len(preds)):
                pred = preds[i]
                true = (y[i][mask[i]].cpu().tolist())
                all_preds.extend(pred)
                all_labels.extend(true)

    print(classification_report(all_labels, all_preds, target_names=[id2label[i] for i in sorted(id2label)], digits=4))
    print("Micro F1:", f1_score(all_labels, all_preds, average="micro"))
    print("Macro F1:", f1_score(all_labels, all_preds, average="macro"))

In [ ]:
evaluate_test_set(model=trained_model,test_dataset=test_data,label2id=label2id,batch_size=64,pad_label=-100)

100%|██████████| 34/34 [00:07<00:00,  4.38it/s]


                    precision    recall  f1-score   support

     B-ACCOUNTNAME     0.9630    0.9905    0.9765       105
   B-ACCOUNTNUMBER     0.9818    0.9730    0.9774       111
B-CREDITCARDNUMBER     0.7864    0.9101    0.8438        89
           B-EMAIL     0.9806    0.9744    0.9775       156
            B-IPV4     0.7597    0.9286    0.8357       126
            B-IPV6     0.8532    0.8692    0.8611       107
             B-MAC     1.0000    1.0000    1.0000        84
        B-PASSWORD     0.9794    0.9314    0.9548       102
    B-PHONE_NUMBER     1.0000    1.0000    1.0000       107
             B-SSN     1.0000    1.0000    1.0000        93
        B-USERNAME     0.8397    0.7483    0.7914       147
     I-ACCOUNTNAME     0.9622    0.9944    0.9780       179
   I-ACCOUNTNUMBER     0.9922    0.9845    0.9884       388
I-CREDITCARDNUMBER     0.7818    0.9169    0.8440       758
           I-EMAIL     0.9898    0.9992    0.9945      1258
            I-IPV4     0.7597    0.9286

In [ ]:
checkpoint = {"model_state_dict": model.state_dict(),
    "label2id": label2id,
    "id2label": {v: k for k, v in label2id.items()},
    }

torch.save(checkpoint, "pii_ner_model.pth")
print("Model saved successfully as 'pii_ner_model.pth'")

Model saved successfully as 'pii_ner_model.pth'
